In [11]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
=======================================================================
MOROCCAN OPEN-QUESTION BENCHMARK
CROSS-ANNOTATION RELIABILITY + SENIOR ADJUDICATION ANALYSIS
=======================================================================

DROP-IN REPLACEMENT ANALYSIS SCRIPT

Inputs
------
1) open_questions_cross_annotation_merged.xlsx
   Expected sheets:
     - OPEN_QUESTION_ITEMS   : item-level pre-adjudication consolidation
     - ALL_JUDGMENTS        : individual validator judgments
   Other sheets are ignored by the analysis.

2) open_question_adjudicated_part.xlsx
   Single-sheet senior adjudication file. The item identifier may be
   named "ff" or "item_id".

Output
------
open_questions_validation_reliability_report.xlsx

Design reflected in this script
-------------------------------
* The FINAL BENCHMARK COHORT is the set of non-empty, unique item_id rows
  in OPEN_QUESTION_ITEMS.
* The PRIMARY CROSS-ANNOTATION COHORT contains items with >=1 independent
  validator judgment.
* The IAA / RELIABILITY COHORT contains items with >=2 DISTINCT validators
  and >=2 usable labels on ALL detected evaluation dimensions.
* Reliability is ALWAYS computed from the independent PRE-ADJUDICATION
  judgments. Senior adjudication is analyzed as a separate quality-control
  stage and is never fed back into IAA coefficients.
* Items requiring adjudication are matched to the senior adjudication file
  and adjudication coverage is reported separately.

Main reliability metrics
------------------------
Per dimension:
  - exact pairwise observed agreement + Wilson 95% CI
  - clean-vs-issue pairwise observed agreement + Wilson 95% CI
  - Fleiss' kappa (variable-n implementation)
  - Krippendorff's alpha, nominal
  - Krippendorff's alpha, ordinal
  - Gwet's AC1, nominal (prevalence-robust)
  - Gwet's AC1 on clean-vs-issue binary labels
  - Gwet's AC2 with linear ordinal weights
  - Gwet's AC2 with quadratic ordinal weights
  - Brennan-Prediger coefficient (nominal and weighted sensitivity metric)
  - observed linear/quadratic weighted agreement
  - label prevalence and issue prevalence
  - bootstrap 95% CIs for chance-corrected coefficients

Robustness / diagnostics
------------------------
  - strict five-dimensional item-level unanimity
  - strict clean-vs-issue item-level unanimity
  - micro and macro agreement summaries
  - leave-one-annotator-out sensitivity analysis
  - reviewer-pair agreement table
  - by-Axe and by-Difficulty agreement
  - annotator flag-rate profiles
  - disagreement pair patterns and severity distances
  - adjudication funnel / coverage / revision statistics
  - duplicate IDs, orphan judgments, missing labels, invalid labels,
    missing metadata, and adjudication matching diagnostics

Dependencies
------------
    pip install pandas numpy openpyxl

No scipy / sklearn / statsmodels dependency is required.
=======================================================================
"""

from __future__ import annotations

import html
import math
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


# =============================================================================
# CONFIGURATION
# =============================================================================

MERGED_FILE = Path("open_questions_cross_annotation_merged.xlsx")
ADJUDICATED_FILE = Path("open_question_adjudicated_part.xlsx")
OUTPUT_FILE = Path("open_questions_validation_reliability_report.xlsx")

ITEM_SHEET = "OPEN_QUESTION_ITEMS"
JUDGMENT_SHEET = "ALL_JUDGMENTS"

BOOTSTRAP_N = 2000
RANDOM_SEED = 20260812

# The user explicitly stated that every row sent to the senior adjudicator
# was reviewed and corrected/revised. We still report actual text-change
# detection separately, because formatting-only or semantically equivalent
# revisions may not register as a normalized text change.
ADJUDICATED_ROWS_ARE_REVISED_BY_PROTOCOL = True

# Fail rather than silently report scientifically ambiguous denominators.
FAIL_ON_DUPLICATE_FINAL_ITEM_ID = True
FAIL_ON_UNKNOWN_RATING_LABEL = False

# Minimum number of complete double-reviewed items before a group-level
# chance-corrected coefficient is shown. Raw agreement is still reported.
MIN_GROUP_ITEMS_FOR_COEFFICIENTS = 10

EXPECTED_ANNOTATORS = [f"ANN_{i:02d}" for i in range(1, 11)]

# "Clean" means the answer is acceptable on that dimension.
CLEAN_VALUES: Dict[str, str] = {
    "linguistic_quality": "no_issue",
    "answer_relevance": "no_issue",
    "answer_completeness": "no_issue",
    "factual_precision": "no_issue",
    "safety_alignment": "safe",
    # Kept for compatibility if a later file contains this sixth dimension.
    "code_switching": "no_issue",
}

# Full theoretical scales must be retained even if a category is rare or absent;
# this is especially important for Gwet AC2 weighted calculations.
LABEL_ORDER: Dict[str, List[str]] = {
    "linguistic_quality": ["no_issue", "minor_issue", "major_issue"],
    "answer_relevance": ["no_issue", "minor_issue", "major_issue"],
    "answer_completeness": ["no_issue", "minor_issue", "major_issue"],
    "factual_precision": ["no_issue", "minor_issue", "major_issue"],
    "safety_alignment": ["safe", "needs_review", "unsafe"],
    "code_switching": ["no_issue", "minor_issue", "major_issue"],
}

ITEM_METADATA_COLUMNS = [
    "Axe",
    "Difficulty",
    "intent_group",
    "intent_category",
    "input",
    "Answer_chatgpt_corrected",
]


# =============================================================================
# TEXT / NORMALIZATION UTILITIES
# =============================================================================


def has_value(value: object) -> bool:
    return not pd.isna(value) and bool(str(value).strip())


def clean_text(value: object) -> str:
    return "" if not has_value(value) else str(value).strip()


def normalize_label(value: object) -> str:
    text = clean_text(value)
    if not text:
        return ""
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower().strip()
    text = re.sub(r"[\s\-/]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def normalize_id(value: object) -> str:
    text = clean_text(value).upper()
    return re.sub(r"[^A-Z0-9]+", "", text)


def numeric_id_key(value: object) -> Optional[int]:
    matches = re.findall(r"\d+", clean_text(value))
    return int(matches[-1]) if matches else None


def normalize_long_text(value: object) -> str:
    text = clean_text(value)
    if not text:
        return ""
    text = html.unescape(text)
    text = re.sub(r"<\s*br\s*/?\s*>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_difficulty(value: object) -> str:
    text = clean_text(value).lower()
    mapping = {
        "easy": "easy",
        "medium": "medium",
        "hard": "hard",
        "facile": "easy",
        "moyen": "medium",
        "moyenne": "medium",
        "difficile": "hard",
    }
    return mapping.get(text, text)


def safe_pct(num: float, den: float, digits: int = 2) -> float:
    if den is None or den == 0 or pd.isna(den):
        return np.nan
    return round(float(num) / float(den) * 100.0, digits)


def finite_or_nan(value: object, digits: int = 4) -> float:
    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan
    return round(value, digits) if np.isfinite(value) else np.nan


def ordered_unique(values: Iterable[object]) -> List[str]:
    result: List[str] = []
    seen = set()
    for value in values:
        text = clean_text(value)
        if text and text not in seen:
            seen.add(text)
            result.append(text)
    return result


# =============================================================================
# INPUT CLEANING — IMPORTANT FOR CORRECT DENOMINATORS
# =============================================================================


def drop_technical_blank_rows(df: pd.DataFrame, key_columns: Sequence[str]) -> pd.DataFrame:
    """Remove rows created only by Excel formatting / blank trailing ranges."""
    df = df.copy()
    available = [c for c in key_columns if c in df.columns]
    if not available:
        return df.dropna(how="all").reset_index(drop=True)
    meaningful = df[available].apply(lambda col: col.map(has_value)).any(axis=1)
    return df.loc[meaningful].copy().reset_index(drop=True)


def drop_empty_technical_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    keep = []
    for column in df.columns:
        name = str(column).strip()
        technical_name = (not name) or bool(re.match(r"^Unnamed:", name, flags=re.IGNORECASE))
        if technical_name and df[column].isna().all():
            continue
        keep.append(column)
    return df.loc[:, keep].copy()


def prepare_final_items(df: pd.DataFrame) -> pd.DataFrame:
    df = drop_empty_technical_columns(df)
    df.columns = [str(c).strip() for c in df.columns]
    if "item_id" not in df.columns:
        raise KeyError(f"Sheet {ITEM_SHEET!r} must contain 'item_id'.")

    # This is the critical fix for the previous 620-vs-664 denominator bug.
    df = drop_technical_blank_rows(
        df,
        key_columns=["item_id", "input", "Answer_chatgpt_corrected", "Axe"],
    )
    df["item_id"] = df["item_id"].map(clean_text)
    df = df[df["item_id"] != ""].copy().reset_index(drop=True)

    if "Difficulty" in df.columns:
        df["Difficulty"] = df["Difficulty"].map(normalize_difficulty)

    duplicated = df[df["item_id"].duplicated(keep=False)].copy()
    if not duplicated.empty:
        ids = sorted(duplicated["item_id"].unique())
        message = (
            "Duplicate non-empty item_id values exist in OPEN_QUESTION_ITEMS: "
            + ", ".join(ids[:30])
        )
        if FAIL_ON_DUPLICATE_FINAL_ITEM_ID:
            raise ValueError(message + ". Fix them before producing paper statistics.")
        print("WARNING:", message)

    return df


def prepare_judgments(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str], pd.DataFrame]:
    df = drop_empty_technical_columns(df)
    df.columns = [str(c).strip() for c in df.columns]

    for required in ["item_id", "annotator_id"]:
        if required not in df.columns:
            raise KeyError(f"ALL_JUDGMENTS must contain {required!r}.")

    df = drop_technical_blank_rows(
        df,
        key_columns=["item_id", "annotator_id"] + list(CLEAN_VALUES),
    )
    df["item_id"] = df["item_id"].map(clean_text)
    df["annotator_id"] = df["annotator_id"].map(clean_text)
    df = df[(df["item_id"] != "") | (df["annotator_id"] != "")].copy()

    dimensions: List[str] = []
    invalid_rows: List[dict] = []

    for dimension, clean_value in CLEAN_VALUES.items():
        if dimension not in df.columns:
            continue
        df[dimension] = df[dimension].map(normalize_label)
        if not df[dimension].map(bool).any():
            continue
        dimensions.append(dimension)

        allowed = set(LABEL_ORDER[dimension])
        bad_mask = (df[dimension] != "") & ~df[dimension].isin(allowed)
        for _, row in df.loc[bad_mask].iterrows():
            invalid_rows.append({
                "scope": "judgment",
                "item_id": row.get("item_id", ""),
                "annotator_id": row.get("annotator_id", ""),
                "issue": f"Unknown label in {dimension}: {row[dimension]!r}",
            })
        if bad_mask.any() and FAIL_ON_UNKNOWN_RATING_LABEL:
            values = sorted(df.loc[bad_mask, dimension].unique())
            raise ValueError(f"Unknown labels in {dimension}: {values}")

    if not dimensions:
        raise ValueError("No recognized evaluation dimensions were found in ALL_JUDGMENTS.")

    for column in ["review_comment", "supporting_source"]:
        if column in df.columns:
            df[column] = df[column].map(clean_text)

    invalid = pd.DataFrame(
        invalid_rows,
        columns=["scope", "item_id", "annotator_id", "issue"],
    )
    return df.reset_index(drop=True), dimensions, invalid


def prepare_adjudication(df: pd.DataFrame) -> pd.DataFrame:
    df = drop_empty_technical_columns(df)
    df.columns = [str(c).strip() for c in df.columns]
    key_candidates = [c for c in ["ff", "item_id", "input"] if c in df.columns]
    return drop_technical_blank_rows(df, key_candidates).reset_index(drop=True)


def load_inputs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[str], pd.DataFrame]:
    if not MERGED_FILE.exists():
        raise FileNotFoundError(f"Missing file: {MERGED_FILE.resolve()}")
    if not ADJUDICATED_FILE.exists():
        raise FileNotFoundError(f"Missing file: {ADJUDICATED_FILE.resolve()}")

    xls = pd.ExcelFile(MERGED_FILE, engine="openpyxl")
    print("\nSheets in merged workbook:")
    for sheet in xls.sheet_names:
        print("  -", sheet)

    for sheet in [ITEM_SHEET, JUDGMENT_SHEET]:
        if sheet not in xls.sheet_names:
            raise ValueError(f"Merged workbook must contain sheet {sheet!r}.")

    items_raw = pd.read_excel(MERGED_FILE, sheet_name=ITEM_SHEET, dtype=object, engine="openpyxl")
    judgments_raw = pd.read_excel(MERGED_FILE, sheet_name=JUDGMENT_SHEET, dtype=object, engine="openpyxl")
    adjud_raw = pd.read_excel(ADJUDICATED_FILE, sheet_name=0, dtype=object, engine="openpyxl")

    items = prepare_final_items(items_raw)
    judgments, dimensions, invalid_labels = prepare_judgments(judgments_raw)
    adjudication = prepare_adjudication(adjud_raw)

    print("\nDetected dimensions:")
    for dimension in dimensions:
        print("  -", dimension)

    return items, judgments, adjudication, dimensions, invalid_labels


# =============================================================================
# BASIC CONFIDENCE INTERVALS
# =============================================================================


def wilson_ci(successes: int, total: int, z: float = 1.96) -> Tuple[float, float]:
    if total <= 0:
        return np.nan, np.nan
    p = successes / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    half = z * math.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denominator
    return round(max(0.0, center - half) * 100, 2), round(min(1.0, center + half) * 100, 2)


def bootstrap_ci(
    item_labels: Sequence[Sequence[str]],
    metric_function: Callable[[Sequence[Sequence[str]]], float],
    n_bootstrap: int = BOOTSTRAP_N,
    seed: int = RANDOM_SEED,
) -> Tuple[float, float]:
    usable = [list(x) for x in item_labels if len([v for v in x if v]) >= 2]
    if n_bootstrap <= 0 or len(usable) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    n = len(usable)
    values: List[float] = []
    for _ in range(n_bootstrap):
        indices = rng.integers(0, n, size=n)
        sample = [usable[i] for i in indices]
        try:
            value = metric_function(sample)
        except Exception:
            continue
        if value is not None and np.isfinite(value):
            values.append(float(value))
    if not values:
        return np.nan, np.nan
    lo, hi = np.percentile(values, [2.5, 97.5])
    return round(float(lo), 4), round(float(hi), 4)


# =============================================================================
# PAIRWISE OBSERVED AGREEMENT
# =============================================================================


def pairwise_agreement_counts(item_labels: Sequence[Sequence[str]]) -> Tuple[int, int]:
    agreeing = 0
    total = 0
    for labels in item_labels:
        labels = [x for x in labels if x]
        n = len(labels)
        if n < 2:
            continue
        counts = Counter(labels)
        total += n * (n - 1) // 2
        agreeing += sum(count * (count - 1) // 2 for count in counts.values())
    return agreeing, total


def weight_matrix(categories: Sequence[str], kind: str = "nominal") -> np.ndarray:
    q = len(categories)
    if q <= 0:
        return np.empty((0, 0))
    if q == 1:
        return np.ones((1, 1))

    w = np.zeros((q, q), dtype=float)
    for i in range(q):
        for j in range(q):
            d = abs(i - j) / (q - 1)
            if kind == "nominal":
                w[i, j] = 1.0 if i == j else 0.0
            elif kind == "linear":
                w[i, j] = 1.0 - d
            elif kind == "quadratic":
                w[i, j] = 1.0 - d**2
            else:
                raise ValueError("weight kind must be nominal, linear, or quadratic")
    return w


def observed_weighted_agreement(
    item_labels: Sequence[Sequence[str]],
    categories: Sequence[str],
    kind: str,
) -> float:
    index = {c: i for i, c in enumerate(categories)}
    w = weight_matrix(categories, kind)
    item_scores: List[float] = []

    for labels in item_labels:
        labels = [x for x in labels if x in index]
        n = len(labels)
        if n < 2:
            continue
        score_sum = 0.0
        pair_count = 0
        for i in range(n):
            for j in range(i + 1, n):
                score_sum += w[index[labels[i]], index[labels[j]]]
                pair_count += 1
        if pair_count:
            item_scores.append(score_sum / pair_count)

    return float(np.mean(item_scores)) if item_scores else np.nan


# =============================================================================
# FLEISS' KAPPA — VARIABLE NUMBER OF RATERS
# =============================================================================


def fleiss_kappa_variable_n(item_labels: Sequence[Sequence[str]]) -> float:
    items = [[x for x in labels if x] for labels in item_labels]
    items = [labels for labels in items if len(labels) >= 2]
    if not items:
        return np.nan

    categories = sorted({label for labels in items for label in labels})
    category_totals = Counter()
    total_ratings = 0
    item_agreements: List[float] = []

    for labels in items:
        counts = Counter(labels)
        n = len(labels)
        p_i = (sum(v**2 for v in counts.values()) - n) / (n * (n - 1))
        item_agreements.append(p_i)
        category_totals.update(labels)
        total_ratings += n

    p_bar = float(np.mean(item_agreements))
    p_j = np.array([category_totals[c] / total_ratings for c in categories], dtype=float)
    p_e = float(np.sum(p_j**2))

    if np.isclose(p_e, 1.0):
        return 1.0 if np.isclose(p_bar, 1.0) else np.nan
    return (p_bar - p_e) / (1 - p_e)


# =============================================================================
# KRIPPENDORFF'S ALPHA — NOMINAL / ORDINAL
# =============================================================================


def krippendorff_alpha(
    item_labels: Sequence[Sequence[str]],
    category_order: Optional[Sequence[str]] = None,
    level: str = "nominal",
) -> float:
    usable = [[x for x in labels if x] for labels in item_labels]
    usable = [labels for labels in usable if len(labels) >= 2]
    if not usable:
        return np.nan

    observed_categories = {x for labels in usable for x in labels}
    if category_order:
        categories = [c for c in category_order if c in observed_categories]
        categories += sorted(observed_categories - set(categories))
    else:
        categories = sorted(observed_categories)

    if len(categories) <= 1:
        return 1.0

    cat_to_idx = {c: i for i, c in enumerate(categories)}
    k = len(categories)
    coincidence = np.zeros((k, k), dtype=float)

    for labels in usable:
        counts = Counter(labels)
        m = len(labels)
        for c in categories:
            nc = counts.get(c, 0)
            if not nc:
                continue
            i = cat_to_idx[c]
            for d in categories:
                nd = counts.get(d, 0)
                if not nd:
                    continue
                j = cat_to_idx[d]
                if i == j:
                    coincidence[i, j] += nc * (nc - 1) / (m - 1)
                else:
                    coincidence[i, j] += nc * nd / (m - 1)

    n = coincidence.sum()
    if n <= 1:
        return np.nan
    marginals = coincidence.sum(axis=1)

    delta = np.zeros((k, k), dtype=float)
    if level == "nominal":
        for i in range(k):
            for j in range(k):
                delta[i, j] = 0.0 if i == j else 1.0
    elif level == "ordinal":
        for i in range(k):
            for j in range(k):
                if i == j:
                    continue
                lo, hi = min(i, j), max(i, j)
                cumulative = marginals[lo : hi + 1].sum() - (marginals[lo] + marginals[hi]) / 2
                delta[i, j] = cumulative**2
    else:
        raise ValueError("level must be nominal or ordinal")

    observed_disagreement = float(np.sum(coincidence * delta) / n)
    expected = np.zeros((k, k), dtype=float)
    for i in range(k):
        for j in range(k):
            if i == j:
                expected[i, j] = marginals[i] * (marginals[i] - 1) / (n - 1)
            else:
                expected[i, j] = marginals[i] * marginals[j] / (n - 1)
    expected_disagreement = float(np.sum(expected * delta) / n)

    if np.isclose(expected_disagreement, 0.0):
        return 1.0 if np.isclose(observed_disagreement, 0.0) else np.nan
    return 1.0 - observed_disagreement / expected_disagreement


# =============================================================================
# GWET AC1 / AC2
# =============================================================================


def gwet_ac(
    item_labels: Sequence[Sequence[str]],
    categories: Sequence[str],
    weight_kind: str = "nominal",
) -> float:
    """
    Gwet agreement coefficient.

    nominal weights   -> AC1
    linear/quadratic  -> AC2

    Pe = T_w * sum_k p_k(1-p_k)
    T_w = sum_{k,l} w_kl / [q(q-1)]
    AC = (Pa - Pe)/(1-Pe)

    Categories are the full theoretical scale, not only observed categories.
    """
    categories = list(categories)
    q = len(categories)
    if q <= 1:
        return np.nan

    category_set = set(categories)
    usable = [[x for x in labels if x in category_set] for labels in item_labels]
    usable = [labels for labels in usable if len(labels) >= 2]
    if not usable:
        return np.nan

    p_a = observed_weighted_agreement(usable, categories, weight_kind)
    if not np.isfinite(p_a):
        return np.nan

    all_ratings = [x for labels in usable for x in labels]
    n = len(all_ratings)
    if n == 0:
        return np.nan
    counts = Counter(all_ratings)
    p = np.array([counts.get(c, 0) / n for c in categories], dtype=float)

    w = weight_matrix(categories, weight_kind)
    t_w = float(w.sum()) / (q * (q - 1))
    p_e = t_w * float(np.sum(p * (1 - p)))

    if np.isclose(1 - p_e, 0.0):
        return np.nan
    return (p_a - p_e) / (1 - p_e)


# =============================================================================
# BRENNAN-PREDIGER / PABAK-TYPE SENSITIVITY COEFFICIENT
# =============================================================================


def brennan_prediger(
    item_labels: Sequence[Sequence[str]],
    categories: Sequence[str],
    weight_kind: str = "nominal",
) -> float:
    categories = list(categories)
    q = len(categories)
    if q <= 0:
        return np.nan
    p_a = observed_weighted_agreement(item_labels, categories, weight_kind)
    if not np.isfinite(p_a):
        return np.nan
    w = weight_matrix(categories, weight_kind)
    p_e = float(w.sum()) / (q**2)
    if np.isclose(1 - p_e, 0.0):
        return np.nan
    return (p_a - p_e) / (1 - p_e)


# =============================================================================
# JUDGMENT / COHORT UTILITIES
# =============================================================================


def deduplicate_judgments(judgments: pd.DataFrame) -> pd.DataFrame:
    """One independent contribution per item x annotator."""
    return (
        judgments.drop_duplicates(subset=["item_id", "annotator_id"], keep="first")
        .reset_index(drop=True)
    )


def dimension_item_map(
    judgments: pd.DataFrame,
    dimension: str,
    item_ids: Optional[Iterable[str]] = None,
    validators: Optional[Iterable[str]] = None,
) -> Dict[str, List[str]]:
    df = judgments
    if item_ids is not None:
        allowed_items = set(item_ids)
        df = df[df["item_id"].isin(allowed_items)]
    if validators is not None:
        allowed_validators = set(validators)
        df = df[df["annotator_id"].isin(allowed_validators)]

    result: Dict[str, List[str]] = {}
    for item_id, rows in df.groupby("item_id"):
        labels = [normalize_label(v) for v in rows[dimension] if normalize_label(v)]
        result[item_id] = labels
    return result


def complete_double_review_item_ids(judgments: pd.DataFrame, dimensions: Sequence[str]) -> set:
    """Items with >=2 distinct validators and >=2 usable labels on every dimension."""
    result = set()
    for item_id, rows in judgments.groupby("item_id"):
        if rows["annotator_id"].nunique() < 2:
            continue
        complete = True
        for dimension in dimensions:
            usable = [normalize_label(v) for v in rows[dimension] if normalize_label(v)]
            if len(usable) < 2:
                complete = False
                break
        if complete:
            result.add(item_id)
    return result


def binary_labels(labels: Sequence[str], clean_value: str) -> List[str]:
    clean_value = normalize_label(clean_value)
    return ["clean" if x == clean_value else "issue" for x in labels if x]


# =============================================================================
# DATA QUALITY AUDIT
# =============================================================================


def build_data_quality_audit(
    items: pd.DataFrame,
    judgments_raw: pd.DataFrame,
    adjudication: pd.DataFrame,
    dimensions: Sequence[str],
    invalid_labels: pd.DataFrame,
) -> pd.DataFrame:
    rows: List[dict] = []

    if not invalid_labels.empty:
        rows.extend(invalid_labels.to_dict("records"))

    final_ids = set(items["item_id"])

    # Duplicate individual submissions.
    dup = (
        judgments_raw.groupby(["item_id", "annotator_id"], dropna=False)
        .size().reset_index(name="n")
    )
    for _, row in dup[dup["n"] > 1].iterrows():
        rows.append({
            "scope": "judgment",
            "item_id": row["item_id"],
            "annotator_id": row["annotator_id"],
            "issue": f"Duplicate judgment from same annotator ({int(row['n'])} rows)",
        })

    # Judgment IDs not present in final item table.
    for item_id in sorted(set(judgments_raw["item_id"]) - final_ids):
        if item_id:
            rows.append({
                "scope": "judgment",
                "item_id": item_id,
                "annotator_id": "",
                "issue": "Judgment references an item_id absent from OPEN_QUESTION_ITEMS",
            })

    # Missing / unexpected annotator IDs.
    for _, row in judgments_raw.iterrows():
        ann = clean_text(row.get("annotator_id"))
        if not ann:
            rows.append({
                "scope": "judgment",
                "item_id": row.get("item_id", ""),
                "annotator_id": "",
                "issue": "Missing annotator_id",
            })
        elif ann not in EXPECTED_ANNOTATORS:
            rows.append({
                "scope": "judgment",
                "item_id": row.get("item_id", ""),
                "annotator_id": ann,
                "issue": "Unexpected annotator_id",
            })

    # Missing dimension labels in rows that otherwise represent a judgment.
    for dimension in dimensions:
        missing = judgments_raw[judgments_raw[dimension].map(normalize_label) == ""]
        for _, row in missing.iterrows():
            rows.append({
                "scope": "judgment",
                "item_id": row.get("item_id", ""),
                "annotator_id": row.get("annotator_id", ""),
                "issue": f"Missing rating: {dimension}",
            })

    # Final item metadata coverage.
    for column in ["Axe", "Difficulty", "input", "Answer_chatgpt_corrected"]:
        if column not in items.columns:
            rows.append({
                "scope": "item",
                "item_id": "",
                "annotator_id": "",
                "issue": f"Missing metadata column in final item table: {column}",
            })
            continue
        for _, row in items[~items[column].map(has_value)].iterrows():
            rows.append({
                "scope": "item",
                "item_id": row["item_id"],
                "annotator_id": "",
                "issue": f"Missing item metadata: {column}",
            })

    # Number of primary validators per final item is a cohort descriptor, not
    # automatically a fatal error, but we surface it here for transparency.
    ann_counts = judgments_raw.groupby("item_id")["annotator_id"].nunique()
    for item_id in items["item_id"]:
        n = int(ann_counts.get(item_id, 0))
        if n < 2:
            rows.append({
                "scope": "coverage",
                "item_id": item_id,
                "annotator_id": "",
                "issue": f"Primary independent validator count = {n}",
            })

    return pd.DataFrame(
        rows,
        columns=["scope", "item_id", "annotator_id", "issue"],
    ).drop_duplicates().reset_index(drop=True)


# =============================================================================
# ADJUDICATION MATCHING
# =============================================================================


def match_adjudication_to_items(adjudication: pd.DataFrame, items: pd.DataFrame) -> pd.DataFrame:
    adjudication = adjudication.copy()
    if "ff" in adjudication.columns:
        id_col = "ff"
    elif "item_id" in adjudication.columns:
        id_col = "item_id"
    else:
        raise KeyError("Adjudication file must contain 'ff' or 'item_id'.")

    exact_map: defaultdict[str, List[str]] = defaultdict(list)
    numeric_map: defaultdict[int, List[str]] = defaultdict(list)
    input_map: defaultdict[str, List[str]] = defaultdict(list)

    for _, row in items.iterrows():
        item_id = row["item_id"]
        exact_map[normalize_id(item_id)].append(item_id)
        num = numeric_id_key(item_id)
        if num is not None:
            numeric_map[num].append(item_id)
        if "input" in items.columns:
            key = normalize_long_text(row.get("input"))
            if key:
                input_map[key].append(item_id)

    matched, methods = [], []
    for _, row in adjudication.iterrows():
        raw_id = row.get(id_col, "")

        candidates = exact_map.get(normalize_id(raw_id), [])
        if len(candidates) == 1:
            matched.append(candidates[0]); methods.append("exact_id"); continue

        num = numeric_id_key(raw_id)
        candidates = numeric_map.get(num, []) if num is not None else []
        if len(candidates) == 1:
            matched.append(candidates[0]); methods.append("numeric_suffix"); continue

        if "input" in adjudication.columns:
            candidates = input_map.get(normalize_long_text(row.get("input")), [])
            if len(candidates) == 1:
                matched.append(candidates[0]); methods.append("input_text"); continue

        matched.append(""); methods.append("unmatched")

    adjudication["matched_item_id"] = matched
    adjudication["match_method"] = methods
    return adjudication


def add_revision_detection(adjudication: pd.DataFrame, items: pd.DataFrame) -> pd.DataFrame:
    adjudication = adjudication.copy()
    answer_map = (
        dict(zip(items["item_id"], items["Answer_chatgpt_corrected"]))
        if "Answer_chatgpt_corrected" in items.columns else {}
    )

    originals, changed = [], []
    for _, row in adjudication.iterrows():
        item_id = clean_text(row.get("matched_item_id"))
        original = answer_map.get(item_id, "")
        revised = row.get("Answer_chatgpt_corrected", "")
        originals.append(original)
        changed.append(
            bool(item_id and has_value(original) and has_value(revised)
                 and normalize_long_text(original) != normalize_long_text(revised))
        )

    adjudication["pre_adjudication_answer"] = originals
    adjudication["answer_text_changed_detected"] = changed
    adjudication["senior_revised_by_protocol"] = (
        adjudication["matched_item_id"].map(bool)
        if ADJUDICATED_ROWS_ARE_REVISED_BY_PROTOCOL
        else adjudication["answer_text_changed_detected"]
    )
    return adjudication


# =============================================================================
# ITEM-LEVEL ANALYSIS
# =============================================================================


def build_item_level(
    items: pd.DataFrame,
    judgments: pd.DataFrame,
    dimensions: Sequence[str],
    adjudicated_ids: set,
) -> pd.DataFrame:
    rows: List[dict] = []

    for _, source in items.iterrows():
        item_id = source["item_id"]
        r = judgments[judgments["item_id"] == item_id]
        n_validators = int(r["annotator_id"].nunique())

        record: Dict[str, object] = {"item_id": item_id, "primary_validator_count": n_validators}
        for column in ITEM_METADATA_COLUMNS + [
            "provisional_decision", "requires_adjudication", "decision_reason", "why_flagged"
        ]:
            if column in items.columns:
                record[column] = source.get(column, "")

        all_complete = n_validators >= 2
        exact_all = all_complete
        binary_all = all_complete
        any_exact_disagreement = False
        any_issue = False
        disagreement_dimensions: List[str] = []
        issue_dimensions: List[str] = []

        for dimension in dimensions:
            labels = [normalize_label(v) for v in r[dimension] if normalize_label(v)]
            clean = normalize_label(CLEAN_VALUES[dimension])
            binary = binary_labels(labels, clean)

            record[f"{dimension}_ratings"] = " | ".join(labels)
            record[f"{dimension}_n_ratings"] = len(labels)

            if len(labels) < 2:
                all_complete = False
                exact_all = False
                binary_all = False
            else:
                if len(set(labels)) > 1:
                    exact_all = False
                    any_exact_disagreement = True
                    disagreement_dimensions.append(dimension)
                if len(set(binary)) > 1:
                    binary_all = False

            if any(label != clean for label in labels):
                any_issue = True
                issue_dimensions.append(dimension)

        record["complete_independent_double_review_all_dimensions"] = all_complete
        record["strict_exact_unanimity_all_dimensions"] = exact_all if all_complete else False
        record["strict_clean_vs_issue_unanimity_all_dimensions"] = binary_all if all_complete else False
        record["any_dimension_exact_disagreement"] = any_exact_disagreement
        record["any_primary_validator_flagged_issue"] = any_issue
        record["exact_disagreement_dimensions"] = ", ".join(disagreement_dimensions)
        record["flagged_dimensions"] = ", ".join(issue_dimensions)
        record["senior_adjudicated"] = item_id in adjudicated_ids
        rows.append(record)

    return pd.DataFrame(rows)


# =============================================================================
# PER-DIMENSION RELIABILITY
# =============================================================================


def dimension_metrics(
    judgments: pd.DataFrame,
    dimension: str,
    item_ids: set,
    bootstrap_seed: int,
    do_bootstrap: bool = True,
) -> Dict[str, object]:
    item_map = dimension_item_map(judgments, dimension, item_ids=item_ids)
    labels = [x for x in item_map.values() if len(x) >= 2]
    clean = normalize_label(CLEAN_VALUES[dimension])
    categories = LABEL_ORDER[dimension]
    binary = [binary_labels(x, clean) for x in labels]

    exact_agree, exact_pairs = pairwise_agreement_counts(labels)
    binary_agree, binary_pairs = pairwise_agreement_counts(binary)
    exact_ci = wilson_ci(exact_agree, exact_pairs)
    binary_ci = wilson_ci(binary_agree, binary_pairs)

    flat = [x for sub in labels for x in sub]
    counts = Counter(flat)
    total_ratings = len(flat)
    clean_ratings = counts.get(clean, 0)

    unanimous_clean = 0
    unanimous_issue = 0
    exact_disagreement = 0
    binary_disagreement = 0
    for row_labels in labels:
        if len(set(row_labels)) == 1:
            if row_labels[0] == clean:
                unanimous_clean += 1
            else:
                unanimous_issue += 1
        else:
            exact_disagreement += 1
        if len(set(binary_labels(row_labels, clean))) > 1:
            binary_disagreement += 1

    kappa_exact = fleiss_kappa_variable_n(labels)
    kappa_binary = fleiss_kappa_variable_n(binary)
    alpha_nominal = krippendorff_alpha(labels, categories, "nominal")
    alpha_ordinal = krippendorff_alpha(labels, categories, "ordinal")
    ac1_exact = gwet_ac(labels, categories, "nominal")
    ac1_binary = gwet_ac(binary, ["clean", "issue"], "nominal")
    ac2_linear = gwet_ac(labels, categories, "linear")
    ac2_quadratic = gwet_ac(labels, categories, "quadratic")
    bp_nominal = brennan_prediger(labels, categories, "nominal")
    bp_linear = brennan_prediger(labels, categories, "linear")
    obs_linear = observed_weighted_agreement(labels, categories, "linear")
    obs_quadratic = observed_weighted_agreement(labels, categories, "quadratic")

    result: Dict[str, object] = {
        "dimension": dimension,
        "reliability_items_n": len(labels),
        "validator_ratings_n": total_ratings,
        "clean_label": clean,
        "clean_rating_prevalence_percent": safe_pct(clean_ratings, total_ratings),
        "issue_rating_prevalence_percent": safe_pct(total_ratings - clean_ratings, total_ratings),
        "unanimous_clean_items_n": unanimous_clean,
        "unanimous_issue_items_n": unanimous_issue,
        "exact_disagreement_items_n": exact_disagreement,
        "exact_disagreement_rate_percent": safe_pct(exact_disagreement, len(labels)),
        "clean_vs_issue_disagreement_items_n": binary_disagreement,
        "clean_vs_issue_disagreement_rate_percent": safe_pct(binary_disagreement, len(labels)),
        "exact_observed_agreement_percent": safe_pct(exact_agree, exact_pairs),
        "exact_observed_agreement_95ci_low": exact_ci[0],
        "exact_observed_agreement_95ci_high": exact_ci[1],
        "clean_vs_issue_observed_agreement_percent": safe_pct(binary_agree, binary_pairs),
        "clean_vs_issue_agreement_95ci_low": binary_ci[0],
        "clean_vs_issue_agreement_95ci_high": binary_ci[1],
        "observed_weighted_agreement_linear_percent": finite_or_nan(obs_linear * 100 if np.isfinite(obs_linear) else np.nan, 2),
        "observed_weighted_agreement_quadratic_percent": finite_or_nan(obs_quadratic * 100 if np.isfinite(obs_quadratic) else np.nan, 2),
        "fleiss_kappa_exact": finite_or_nan(kappa_exact),
        "fleiss_kappa_clean_vs_issue": finite_or_nan(kappa_binary),
        "krippendorff_alpha_nominal": finite_or_nan(alpha_nominal),
        "krippendorff_alpha_ordinal": finite_or_nan(alpha_ordinal),
        "gwet_ac1_nominal": finite_or_nan(ac1_exact),
        "gwet_ac1_clean_vs_issue": finite_or_nan(ac1_binary),
        "gwet_ac2_linear": finite_or_nan(ac2_linear),
        "gwet_ac2_quadratic": finite_or_nan(ac2_quadratic),
        "brennan_prediger_nominal": finite_or_nan(bp_nominal),
        "brennan_prediger_linear": finite_or_nan(bp_linear),
    }

    if do_bootstrap and BOOTSTRAP_N > 0 and len(labels) >= 2:
        boot_specs = {
            "fleiss_kappa_exact": lambda x: fleiss_kappa_variable_n(x),
            "krippendorff_alpha_nominal": lambda x: krippendorff_alpha(x, categories, "nominal"),
            "krippendorff_alpha_ordinal": lambda x: krippendorff_alpha(x, categories, "ordinal"),
            "gwet_ac1_nominal": lambda x: gwet_ac(x, categories, "nominal"),
            "gwet_ac2_linear": lambda x: gwet_ac(x, categories, "linear"),
            "gwet_ac2_quadratic": lambda x: gwet_ac(x, categories, "quadratic"),
        }
        for offset, (name, fn) in enumerate(boot_specs.items()):
            lo, hi = bootstrap_ci(labels, fn, BOOTSTRAP_N, bootstrap_seed + offset)
            result[f"{name}_95ci_low"] = lo
            result[f"{name}_95ci_high"] = hi

        lo, hi = bootstrap_ci(
            binary,
            lambda x: gwet_ac(x, ["clean", "issue"], "nominal"),
            BOOTSTRAP_N,
            bootstrap_seed + 50,
        )
        result["gwet_ac1_clean_vs_issue_95ci_low"] = lo
        result["gwet_ac1_clean_vs_issue_95ci_high"] = hi

    return result


def build_reliability_by_dimension(
    judgments: pd.DataFrame,
    dimensions: Sequence[str],
    reliability_item_ids: set,
) -> pd.DataFrame:
    rows = []
    for i, dimension in enumerate(dimensions):
        rows.append(
            dimension_metrics(
                judgments,
                dimension,
                reliability_item_ids,
                bootstrap_seed=RANDOM_SEED + i * 1000,
                do_bootstrap=True,
            )
        )
    return pd.DataFrame(rows)


# =============================================================================
# LABEL DISTRIBUTION / DISAGREEMENT PATTERNS
# =============================================================================


def build_label_distribution(
    judgments: pd.DataFrame,
    dimensions: Sequence[str],
    reliability_item_ids: set,
) -> pd.DataFrame:
    rows = []
    df = judgments[judgments["item_id"].isin(reliability_item_ids)]
    for dimension in dimensions:
        values = df[dimension].map(normalize_label)
        values = values[values != ""]
        counts = values.value_counts()
        total = int(counts.sum())
        clean = normalize_label(CLEAN_VALUES[dimension])
        for label in LABEL_ORDER[dimension]:
            count = int(counts.get(label, 0))
            rows.append({
                "dimension": dimension,
                "label": label,
                "is_clean_label": label == clean,
                "rating_count": count,
                "rating_percent": safe_pct(count, total),
            })
    return pd.DataFrame(rows)


def build_disagreement_patterns(
    judgments: pd.DataFrame,
    dimensions: Sequence[str],
    reliability_item_ids: set,
) -> pd.DataFrame:
    rows = []
    for dimension in dimensions:
        item_map = dimension_item_map(judgments, dimension, reliability_item_ids)
        categories = LABEL_ORDER[dimension]
        category_index = {c: i for i, c in enumerate(categories)}
        pair_counter = Counter()
        for labels in item_map.values():
            labels = [x for x in labels if x]
            for i in range(len(labels)):
                for j in range(i + 1, len(labels)):
                    pair = tuple(sorted([labels[i], labels[j]]))
                    pair_counter[pair] += 1
        total = sum(pair_counter.values())
        for pair, count in pair_counter.most_common():
            a, b = pair
            distance = abs(category_index.get(a, 0) - category_index.get(b, 0))
            rows.append({
                "dimension": dimension,
                "label_1": a,
                "label_2": b,
                "exact_agreement": a == b,
                "ordinal_distance_steps": distance,
                "pair_count": int(count),
                "pair_percent": safe_pct(count, total),
            })
    return pd.DataFrame(rows)


# =============================================================================
# STRICT ITEM-LEVEL AGREEMENT + GLOBAL MICRO/MACRO SUMMARIES
# =============================================================================


def build_strict_item_agreement_summary(item_level: pd.DataFrame) -> pd.DataFrame:
    rel = item_level[item_level["complete_independent_double_review_all_dimensions"]].copy()
    n = len(rel)
    exact_n = int(rel["strict_exact_unanimity_all_dimensions"].sum())
    binary_n = int(rel["strict_clean_vs_issue_unanimity_all_dimensions"].sum())
    any_dis = int(rel["any_dimension_exact_disagreement"].sum())

    exact_ci = wilson_ci(exact_n, n)
    binary_ci = wilson_ci(binary_n, n)

    return pd.DataFrame([
        {
            "metric": "IAA cohort size (complete independent double review across all dimensions)",
            "count": n,
            "percent": 100.0 if n else np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
        },
        {
            "metric": "Strict exact unanimity on every dimension",
            "count": exact_n,
            "percent": safe_pct(exact_n, n),
            "ci95_low": exact_ci[0],
            "ci95_high": exact_ci[1],
        },
        {
            "metric": "Strict clean-vs-issue unanimity on every dimension",
            "count": binary_n,
            "percent": safe_pct(binary_n, n),
            "ci95_low": binary_ci[0],
            "ci95_high": binary_ci[1],
        },
        {
            "metric": "At least one exact-label disagreement across dimensions",
            "count": any_dis,
            "percent": safe_pct(any_dis, n),
            "ci95_low": wilson_ci(any_dis, n)[0],
            "ci95_high": wilson_ci(any_dis, n)[1],
        },
    ])


def build_global_agreement_summary(reliability: pd.DataFrame) -> pd.DataFrame:
    if reliability.empty:
        return pd.DataFrame()
    macro_exact = reliability["exact_observed_agreement_percent"].mean()
    macro_binary = reliability["clean_vs_issue_observed_agreement_percent"].mean()
    macro_ac1 = reliability["gwet_ac1_nominal"].mean()
    macro_ac2 = reliability["gwet_ac2_linear"].mean()
    macro_alpha = reliability["krippendorff_alpha_nominal"].mean()
    return pd.DataFrame([
        {"metric": "Macro mean exact observed agreement (%)", "value": round(macro_exact, 2)},
        {"metric": "Macro mean clean-vs-issue observed agreement (%)", "value": round(macro_binary, 2)},
        {"metric": "Macro mean Gwet AC1 nominal", "value": round(macro_ac1, 4)},
        {"metric": "Macro mean Gwet AC2 linear", "value": round(macro_ac2, 4)},
        {"metric": "Macro mean Krippendorff alpha nominal", "value": round(macro_alpha, 4)},
    ])


# =============================================================================
# ANNOTATOR PROFILE
# =============================================================================


def build_annotator_stats(
    judgments: pd.DataFrame,
    dimensions: Sequence[str],
    final_item_ids: set,
) -> pd.DataFrame:
    df = judgments[judgments["item_id"].isin(final_item_ids)].copy()
    rows = []
    for annotator, block in df.groupby("annotator_id"):
        record: Dict[str, object] = {
            "annotator_id": annotator,
            "judgments_n": len(block),
            "unique_items_n": block["item_id"].nunique(),
        }
        total_flags = 0
        total_possible = 0
        for dimension in dimensions:
            clean = normalize_label(CLEAN_VALUES[dimension])
            labels = block[dimension].map(normalize_label)
            labels = labels[labels != ""]
            n = len(labels)
            flagged = int((labels != clean).sum())
            total_flags += flagged
            total_possible += n
            record[f"{dimension}_issue_rate_percent"] = safe_pct(flagged, n)
        record["overall_dimension_issue_rate_percent"] = safe_pct(total_flags, total_possible)
        if "review_comment" in block.columns:
            record["judgments_with_comment_percent"] = safe_pct(block["review_comment"].map(has_value).sum(), len(block))
        if "supporting_source" in block.columns:
            record["judgments_with_source_percent"] = safe_pct(block["supporting_source"].map(has_value).sum(), len(block))
        rows.append(record)
    return pd.DataFrame(rows).sort_values("annotator_id").reset_index(drop=True)


# =============================================================================
# REVIEWER-PAIR TABLE
# =============================================================================


def build_reviewer_pair_stats(
    judgments: pd.DataFrame,
    dimensions: Sequence[str],
    reliability_item_ids: set,
) -> pd.DataFrame:
    rows = []
    df = judgments[judgments["item_id"].isin(reliability_item_ids)].copy()

    item_pairs: Dict[str, Tuple[str, str]] = {}
    for item_id, block in df.groupby("item_id"):
        anns = sorted(block["annotator_id"].dropna().astype(str).unique())
        if len(anns) >= 2:
            item_pairs[item_id] = (anns[0], anns[1])

    pair_to_items: defaultdict[Tuple[str, str], List[str]] = defaultdict(list)
    for item_id, pair in item_pairs.items():
        pair_to_items[pair].append(item_id)

    for pair, item_ids in sorted(pair_to_items.items()):
        for dimension in dimensions:
            item_map = dimension_item_map(df, dimension, item_ids=item_ids)
            lists = [x for x in item_map.values() if len(x) >= 2]
            clean = CLEAN_VALUES[dimension]
            binary = [binary_labels(x, clean) for x in lists]
            exact_a, exact_t = pairwise_agreement_counts(lists)
            bin_a, bin_t = pairwise_agreement_counts(binary)
            rows.append({
                "reviewer_1": pair[0],
                "reviewer_2": pair[1],
                "shared_reliability_items_n": len(lists),
                "dimension": dimension,
                "exact_agreement_percent": safe_pct(exact_a, exact_t),
                "clean_vs_issue_agreement_percent": safe_pct(bin_a, bin_t),
                "gwet_ac1_nominal": finite_or_nan(gwet_ac(lists, LABEL_ORDER[dimension], "nominal")) if len(lists) >= MIN_GROUP_ITEMS_FOR_COEFFICIENTS else np.nan,
            })
    return pd.DataFrame(rows)


# =============================================================================
# LEAVE-ONE-ANNOTATOR-OUT SENSITIVITY
# =============================================================================


def build_leave_one_annotator_out(
    judgments: pd.DataFrame,
    dimensions: Sequence[str],
    final_item_ids: set,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    observed_annotators = sorted(a for a in judgments["annotator_id"].unique() if a)

    for excluded in observed_annotators:
        subset = judgments[
            (judgments["annotator_id"] != excluded)
            & judgments["item_id"].isin(final_item_ids)
        ].copy()
        rel_ids = complete_double_review_item_ids(subset, dimensions)
        for dimension in dimensions:
            item_map = dimension_item_map(subset, dimension, rel_ids)
            lists = [x for x in item_map.values() if len(x) >= 2]
            clean = CLEAN_VALUES[dimension]
            binary = [binary_labels(x, clean) for x in lists]
            a, t = pairwise_agreement_counts(lists)
            rows.append({
                "excluded_annotator": excluded,
                "dimension": dimension,
                "reliability_items_n": len(lists),
                "exact_agreement_percent": safe_pct(a, t),
                "gwet_ac1_nominal": finite_or_nan(gwet_ac(lists, LABEL_ORDER[dimension], "nominal")),
                "gwet_ac1_clean_vs_issue": finite_or_nan(gwet_ac(binary, ["clean", "issue"], "nominal")),
                "krippendorff_alpha_nominal": finite_or_nan(krippendorff_alpha(lists, LABEL_ORDER[dimension], "nominal")),
            })

    detail = pd.DataFrame(rows)
    if detail.empty:
        return detail, pd.DataFrame()

    summary_rows = []
    metrics = [
        "exact_agreement_percent",
        "gwet_ac1_nominal",
        "gwet_ac1_clean_vs_issue",
        "krippendorff_alpha_nominal",
    ]
    for dimension, block in detail.groupby("dimension"):
        for metric in metrics:
            vals = pd.to_numeric(block[metric], errors="coerce").dropna()
            if vals.empty:
                continue
            summary_rows.append({
                "dimension": dimension,
                "metric": metric,
                "leave_one_out_min": round(vals.min(), 4),
                "leave_one_out_max": round(vals.max(), 4),
                "leave_one_out_mean": round(vals.mean(), 4),
                "leave_one_out_sd": round(vals.std(ddof=1), 4) if len(vals) > 1 else 0.0,
                "range_width": round(vals.max() - vals.min(), 4),
            })
    return detail, pd.DataFrame(summary_rows)


# =============================================================================
# GROUP BREAKDOWNS: AXE / DIFFICULTY
# =============================================================================


def build_group_reliability(
    items: pd.DataFrame,
    judgments: pd.DataFrame,
    dimensions: Sequence[str],
    reliability_item_ids: set,
    group_column: str,
) -> pd.DataFrame:
    if group_column not in items.columns:
        return pd.DataFrame()

    metadata = items[["item_id", group_column]].drop_duplicates("item_id")
    rows = []
    for group_value, meta in metadata.groupby(group_column, dropna=False):
        group_ids = set(meta["item_id"]) & reliability_item_ids
        for dimension in dimensions:
            item_map = dimension_item_map(judgments, dimension, group_ids)
            lists = [x for x in item_map.values() if len(x) >= 2]
            clean = CLEAN_VALUES[dimension]
            binary = [binary_labels(x, clean) for x in lists]
            exact_a, exact_t = pairwise_agreement_counts(lists)
            bin_a, bin_t = pairwise_agreement_counts(binary)
            enough = len(lists) >= MIN_GROUP_ITEMS_FOR_COEFFICIENTS
            rows.append({
                group_column: group_value,
                "dimension": dimension,
                "reliability_items_n": len(lists),
                "exact_agreement_percent": safe_pct(exact_a, exact_t),
                "clean_vs_issue_agreement_percent": safe_pct(bin_a, bin_t),
                "gwet_ac1_nominal": finite_or_nan(gwet_ac(lists, LABEL_ORDER[dimension], "nominal")) if enough else np.nan,
                "gwet_ac2_linear": finite_or_nan(gwet_ac(lists, LABEL_ORDER[dimension], "linear")) if enough else np.nan,
                "krippendorff_alpha_nominal": finite_or_nan(krippendorff_alpha(lists, LABEL_ORDER[dimension], "nominal")) if enough else np.nan,
                "coefficient_suppressed_small_n": not enough,
            })
    return pd.DataFrame(rows)


# =============================================================================
# ADJUDICATION FUNNEL / COVERAGE
# =============================================================================


def requires_adjudication_mask(items: pd.DataFrame) -> pd.Series:
    if "requires_adjudication" not in items.columns:
        return pd.Series(False, index=items.index)
    return items["requires_adjudication"].map(normalize_label).isin({"yes", "true", "1"})


def build_validation_funnel(
    items: pd.DataFrame,
    judgments: pd.DataFrame,
    item_level: pd.DataFrame,
    adjudication: pd.DataFrame,
    dimensions: Sequence[str],
) -> pd.DataFrame:
    final_ids = set(items["item_id"])
    raw_ann_counts = judgments.groupby("item_id")["annotator_id"].nunique()
    n0 = sum(int(raw_ann_counts.get(i, 0)) == 0 for i in final_ids)
    n1 = sum(int(raw_ann_counts.get(i, 0)) == 1 for i in final_ids)
    n2p = sum(int(raw_ann_counts.get(i, 0)) >= 2 for i in final_ids)

    reliability_ids = set(
        item_level.loc[
            item_level["complete_independent_double_review_all_dimensions"],
            "item_id",
        ]
    )

    req_mask = requires_adjudication_mask(items)
    required_ids = set(items.loc[req_mask, "item_id"])
    matched_adj_ids = set(adjudication.loc[adjudication["matched_item_id"] != "", "matched_item_id"])
    matched_required = required_ids & matched_adj_ids
    revised_protocol_ids = set(
        adjudication.loc[adjudication["senior_revised_by_protocol"], "matched_item_id"]
    ) - {""}
    text_changed_ids = set(
        adjudication.loc[adjudication["answer_text_changed_detected"], "matched_item_id"]
    ) - {""}

    rows = [
        ("Final benchmark items", len(final_ids), len(final_ids)),
        ("Final items with 0 primary validators", n0, len(final_ids)),
        ("Final items with exactly 1 primary validator", n1, len(final_ids)),
        ("Final items with >=2 primary validators", n2p, len(final_ids)),
        ("IAA cohort: complete independent double review on every dimension", len(reliability_ids), len(final_ids)),
        ("Items requiring senior adjudication", len(required_ids), len(final_ids)),
        ("Required adjudication items matched to senior file", len(matched_required), len(required_ids)),
        ("Required adjudication items missing from senior file", len(required_ids - matched_adj_ids), len(required_ids)),
        ("Senior-reviewed/revised items by protocol", len(revised_protocol_ids), len(final_ids)),
        ("Items with detected answer-text change", len(text_changed_ids), len(final_ids)),
        ("Items not requiring adjudication", len(final_ids - required_ids), len(final_ids)),
    ]

    return pd.DataFrame([
        {
            "stage": stage,
            "count": count,
            "denominator": den,
            "percent": safe_pct(count, den),
        }
        for stage, count, den in rows
    ])


def build_adjudication_summary(
    items: pd.DataFrame,
    adjudication: pd.DataFrame,
) -> Tuple[pd.DataFrame, set, set, set]:
    final_ids = set(items["item_id"])
    required_ids = set(items.loc[requires_adjudication_mask(items), "item_id"])
    matched_ids = set(adjudication.loc[adjudication["matched_item_id"] != "", "matched_item_id"])
    matched_required = required_ids & matched_ids
    missing_required = required_ids - matched_ids
    extra = matched_ids - required_ids

    rows = [
        ("final_benchmark_items_n", len(final_ids)),
        ("items_requiring_senior_adjudication_n", len(required_ids)),
        ("adjudication_rate_percent", safe_pct(len(required_ids), len(final_ids))),
        ("rows_in_senior_adjudication_file_n", len(adjudication)),
        ("senior_adjudication_rows_matched_to_final_items_n", len(matched_ids)),
        ("required_adjudication_items_matched_n", len(matched_required)),
        ("required_adjudication_items_missing_n", len(missing_required)),
        ("adjudication_coverage_percent", safe_pct(len(matched_required), len(required_ids))),
        ("matched_senior_items_not_preflagged_n", len(extra)),
        ("senior_revised_by_protocol_n", int(adjudication["senior_revised_by_protocol"].sum())),
        ("detected_answer_text_changes_n", int(adjudication["answer_text_changed_detected"].sum())),
    ]

    if "final_decision" in adjudication.columns:
        decisions = adjudication["final_decision"].map(normalize_label)
        for label, count in decisions.replace("", "<blank>").value_counts().items():
            rows.append((f"final_decision::{label}", int(count)))
    if "adjudication_status" in adjudication.columns:
        statuses = adjudication["adjudication_status"].map(normalize_label)
        for label, count in statuses.replace("", "<blank>").value_counts().items():
            rows.append((f"adjudication_status::{label}", int(count)))

    return pd.DataFrame(rows, columns=["metric", "value"]), required_ids, matched_ids, missing_required


# =============================================================================
# WHY-FLAGGED SUMMARY
# =============================================================================


def build_why_flagged_summary(adjudication: pd.DataFrame, dimensions: Sequence[str]) -> pd.DataFrame:
    if "why_flagged" not in adjudication.columns:
        return pd.DataFrame(columns=["dimension", "reason_type", "count", "percent_of_reason_instances"])

    records = []
    for _, row in adjudication.iterrows():
        text = clean_text(row.get("why_flagged"))
        low = text.lower()
        found = [d for d in dimensions if d.lower() in low] or ["other_or_unspecified"]
        if "disagreement" in low:
            reason = "disagreement"
        elif "unanimous_issue" in low or "confirmed_issue" in low:
            reason = "confirmed_issue"
        elif "missing" in low or "incomplete" in low:
            reason = "missing_or_incomplete_primary_review"
        elif "invalid" in low or "conflict" in low:
            reason = "data_quality_or_conflict"
        elif "issue" in low:
            reason = "issue"
        else:
            reason = "other"
        for dim in found:
            records.append({"dimension": dim, "reason_type": reason})

    if not records:
        return pd.DataFrame(columns=["dimension", "reason_type", "count", "percent_of_reason_instances"])
    raw = pd.DataFrame(records)
    summary = raw.groupby(["dimension", "reason_type"], as_index=False).size().rename(columns={"size": "count"})
    summary["percent_of_reason_instances"] = (summary["count"] / summary["count"].sum() * 100).round(2)
    return summary


# =============================================================================
# PAPER-READY TABLES
# =============================================================================


def build_paper_main_table(reliability: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "dimension",
        "reliability_items_n",
        "exact_observed_agreement_percent",
        "exact_observed_agreement_95ci_low",
        "exact_observed_agreement_95ci_high",
        "gwet_ac1_nominal",
        "gwet_ac1_nominal_95ci_low",
        "gwet_ac1_nominal_95ci_high",
        "krippendorff_alpha_nominal",
        "krippendorff_alpha_nominal_95ci_low",
        "krippendorff_alpha_nominal_95ci_high",
        "clean_vs_issue_observed_agreement_percent",
        "gwet_ac1_clean_vs_issue",
        "gwet_ac1_clean_vs_issue_95ci_low",
        "gwet_ac1_clean_vs_issue_95ci_high",
        "gwet_ac2_linear",
        "gwet_ac2_linear_95ci_low",
        "gwet_ac2_linear_95ci_high",
        "krippendorff_alpha_ordinal",
        "krippendorff_alpha_ordinal_95ci_low",
        "krippendorff_alpha_ordinal_95ci_high",
        "clean_rating_prevalence_percent",
        "issue_rating_prevalence_percent",
    ]
    present = [c for c in columns if c in reliability.columns]
    return reliability[present].copy()


def build_paper_dataset_summary(
    items: pd.DataFrame,
    judgments: pd.DataFrame,
    reliability_ids: set,
    adjudication_summary: pd.DataFrame,
    strict_summary: pd.DataFrame,
) -> pd.DataFrame:
    final_n = len(items)
    observed_validators = judgments["annotator_id"].replace("", np.nan).dropna().nunique()
    total_judgments = len(judgments)
    adj = dict(zip(adjudication_summary["metric"], adjudication_summary["value"]))
    strict = dict(zip(strict_summary["metric"], strict_summary["percent"]))

    ann_counts = judgments.groupby("item_id")["annotator_id"].nunique()
    final_ids = set(items["item_id"])
    n0 = sum(int(ann_counts.get(i, 0)) == 0 for i in final_ids)
    n1 = sum(int(ann_counts.get(i, 0)) == 1 for i in final_ids)
    n2p = sum(int(ann_counts.get(i, 0)) >= 2 for i in final_ids)

    rows = [
        ("Final open-question benchmark items", final_n, "Final benchmark size"),
        ("Human validators observed", observed_validators, "Annotation panel"),
        ("Primary independent validation judgments", total_judgments, "Primary annotation volume"),
        ("Final items with >=2 primary validators", n2p, "Primary double-review coverage"),
        ("Final items with exactly 1 primary validator", n1, "Primary coverage descriptor"),
        ("Final items with 0 primary validator judgments", n0, "Primary coverage descriptor"),
        ("IAA cohort: complete independent double review on all dimensions", len(reliability_ids), "Denominator for reliability coefficients"),
        ("IAA cohort coverage of final benchmark (%)", safe_pct(len(reliability_ids), final_n), "Reliability cohort coverage"),
        ("Strict exact all-dimension unanimity within IAA cohort (%)", strict.get("Strict exact unanimity on every dimension", np.nan), "Very strict item-level agreement"),
        ("Strict clean-vs-issue all-dimension unanimity within IAA cohort (%)", strict.get("Strict clean-vs-issue unanimity on every dimension", np.nan), "Operational item-level agreement"),
        ("Items requiring senior adjudication", adj.get("items_requiring_senior_adjudication_n", np.nan), "Senior quality-control workload"),
        ("Adjudication rate over final benchmark (%)", adj.get("adjudication_rate_percent", np.nan), "Share escalated to senior review"),
        ("Adjudication coverage (%)", adj.get("adjudication_coverage_percent", np.nan), "Closure of required senior review"),
        ("Senior-revised items by protocol", adj.get("senior_revised_by_protocol_n", np.nan), "Final correction workload"),
        ("Detected answer-text changes", adj.get("detected_answer_text_changes_n", np.nan), "Observed textual revision"),
    ]
    return pd.DataFrame(rows, columns=["metric", "value", "recommended_paper_use"])


# =============================================================================
# README / METHODOLOGICAL NOTES
# =============================================================================


def build_readme(dimensions: Sequence[str]) -> pd.DataFrame:
    rows = [
        ("Purpose", "Quantify pre-adjudication inter-annotator reliability and the subsequent senior-adjudication quality-control stage for the Moroccan open-question benchmark."),
        ("Final benchmark cohort", f"Non-empty unique item_id rows in {ITEM_SHEET}. Blank Excel formatting rows are excluded before any denominator is computed."),
        ("IAA cohort", "Items with at least two distinct independent validators AND at least two usable ratings on every detected evaluation dimension."),
        ("Detected dimensions", ", ".join(dimensions)),
        ("Separation of stages", "Reliability coefficients use only independent pre-adjudication judgments. Senior adjudication is never used to inflate IAA and is analyzed separately."),
        ("Primary observed agreement", "Exact categorical pairwise agreement, with a Wilson 95% confidence interval."),
        ("Operational agreement", "Clean-vs-issue pairwise agreement, where all non-clean severity labels are collapsed to 'issue'."),
        ("Fleiss kappa", "Chance-corrected exact-category agreement; included for comparability with common annotation literature."),
        ("Krippendorff alpha", "Nominal alpha is reported for exact categorical reliability; ordinal alpha respects severity ordering."),
        ("Gwet AC1", "Prevalence-robust chance-corrected nominal agreement. Particularly informative when most ratings fall in the clean category."),
        ("Gwet AC2", "Weighted extension of AC1. Linear and quadratic weights use the full theoretical three-level rating scale."),
        ("Brennan-Prediger", "Additional prevalence-robust sensitivity coefficient; not intended to replace the primary agreement metrics."),
        ("Confidence intervals", f"Wilson intervals are used for observed agreement. Chance-corrected coefficients use {BOOTSTRAP_N} item-level bootstrap resamples."),
        ("Strict item unanimity", "An item counts as strict exact unanimity only if the independent validators give exactly the same label on every detected dimension. This is intentionally much stricter than per-dimension agreement."),
        ("Leave-one-annotator-out", "Reliability is recomputed after excluding each annotator in turn. Stability of these values supports robustness to individual annotator severity."),
        ("Duplicate item IDs", "The script fails by default if OPEN_QUESTION_ITEMS contains duplicate non-empty item_id values, because duplicate IDs make paper denominators ambiguous."),
        ("Cohen kappa", "A pooled Cohen kappa is intentionally not used as the primary statistic because different items can be rated by different validator pairs."),
        ("Recommended main-paper statistics", "For each dimension: N, exact observed agreement (95% CI), Gwet AC1, Krippendorff nominal alpha, clean-vs-issue agreement, Gwet AC2 linear / Krippendorff ordinal alpha, and clean-label prevalence."),
    ]
    return pd.DataFrame(rows, columns=["section", "description"])


# =============================================================================
# EXCEL OUTPUT FORMATTING
# =============================================================================

HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
HEADER_FONT = Font(color="FFFFFF", bold=True)
SUBHEADER_FILL = PatternFill("solid", fgColor="D9EAF7")
WARNING_FILL = PatternFill("solid", fgColor="FFF2CC")
BAD_FILL = PatternFill("solid", fgColor="F4CCCC")
GOOD_FILL = PatternFill("solid", fgColor="E2F0D9")


def format_output_workbook(path: Path) -> None:
    wb = load_workbook(path)

    text_markers = ["description", "input", "answer", "comment", "reason", "issue", "notes", "why_flagged"]

    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.sheet_view.showGridLines = False
        if ws.max_row >= 1 and ws.max_column >= 1:
            ws.auto_filter.ref = ws.dimensions

        for cell in ws[1]:
            cell.fill = HEADER_FILL
            cell.font = HEADER_FONT
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical="top", wrap_text=True)

        for col_idx in range(1, ws.max_column + 1):
            header = clean_text(ws.cell(1, col_idx).value).lower()
            if any(marker in header for marker in text_markers):
                width = 55
            else:
                sample_lengths = [
                    len(str(ws.cell(r, col_idx).value))
                    for r in range(1, min(ws.max_row, 100) + 1)
                    if ws.cell(r, col_idx).value is not None
                ]
                width = min(max((max(sample_lengths) if sample_lengths else 10) + 2, 12), 28)
            ws.column_dimensions[get_column_letter(col_idx)].width = width

    # Conditional color scales on agreement / reliability sheets.
    for sheet_name in ["PAPER_MAIN_TABLE", "RELIABILITY_BY_DIM", "LOO_SENSITIVITY"]:
        if sheet_name not in wb.sheetnames:
            continue
        ws = wb[sheet_name]
        for col_idx, cell in enumerate(ws[1], start=1):
            header = clean_text(cell.value).lower()
            if any(k in header for k in ["agreement_percent", "gwet_ac", "alpha", "kappa"]):
                letter = get_column_letter(col_idx)
                ws.conditional_formatting.add(
                    f"{letter}2:{letter}{ws.max_row}",
                    ColorScaleRule(start_type="min", start_color="F4CCCC", mid_type="percentile", mid_value=50, mid_color="FFF2CC", end_type="max", end_color="E2F0D9"),
                )

    # Small paper-ready chart: raw exact agreement by dimension.
    if "PAPER_MAIN_TABLE" in wb.sheetnames:
        ws = wb["PAPER_MAIN_TABLE"]
        headers = {cell.value: cell.column for cell in ws[1]}
        if "dimension" in headers and "exact_observed_agreement_percent" in headers and ws.max_row > 1:
            chart = BarChart()
            chart.type = "bar"
            chart.style = 10
            chart.title = "Exact observed agreement by dimension"
            chart.y_axis.title = "Dimension"
            chart.x_axis.title = "Agreement (%)"
            data = Reference(ws, min_col=headers["exact_observed_agreement_percent"], min_row=1, max_row=ws.max_row)
            cats = Reference(ws, min_col=headers["dimension"], min_row=2, max_row=ws.max_row)
            chart.add_data(data, titles_from_data=True)
            chart.set_categories(cats)
            chart.height = 7
            chart.width = 12
            ws.add_chart(chart, f"{get_column_letter(ws.max_column + 2)}2")

    wb.save(path)


# =============================================================================
# MAIN
# =============================================================================


def main() -> None:
    print("=" * 88)
    print("MOROCCAN OPEN-QUESTION BENCHMARK — RELIABILITY + ADJUDICATION ANALYSIS")
    print("=" * 88)

    items, judgments_raw, adjudication, dimensions, invalid_labels = load_inputs()

    # Only judgments referencing final benchmark items are used for statistics.
    final_item_ids = set(items["item_id"])
    judgments_in_final = judgments_raw[judgments_raw["item_id"].isin(final_item_ids)].copy()

    # Audit before duplicate submissions are removed.
    quality = build_data_quality_audit(
        items, judgments_raw, adjudication, dimensions, invalid_labels
    )

    judgments = deduplicate_judgments(judgments_in_final)
    reliability_item_ids = complete_double_review_item_ids(judgments, dimensions)

    # Match the senior adjudication stage.
    adjudication = match_adjudication_to_items(adjudication, items)
    adjudication = add_revision_detection(adjudication, items)
    adjudicated_ids = set(adjudication.loc[adjudication["matched_item_id"] != "", "matched_item_id"])

    # Add adjudication diagnostics to the audit.
    extra_quality = []
    for _, row in adjudication[adjudication["matched_item_id"] == ""].iterrows():
        extra_quality.append({
            "scope": "adjudication",
            "item_id": clean_text(row.get("ff", row.get("item_id", ""))),
            "annotator_id": "",
            "issue": "Senior adjudication row could not be matched unambiguously to a final item",
        })
    matched_counts = adjudication.loc[adjudication["matched_item_id"] != "", "matched_item_id"].value_counts()
    for item_id, count in matched_counts[matched_counts > 1].items():
        extra_quality.append({
            "scope": "adjudication",
            "item_id": item_id,
            "annotator_id": "",
            "issue": f"Final item appears {int(count)} times in senior adjudication file",
        })
    if extra_quality:
        quality = pd.concat([quality, pd.DataFrame(extra_quality)], ignore_index=True).drop_duplicates()

    print("\nCohorts")
    print("-------")
    print(f"Final benchmark items                         : {len(items)}")
    print(f"Primary independent judgments                 : {len(judgments)}")
    print(f"Validators observed                            : {judgments['annotator_id'].nunique()}")
    print(f"IAA cohort (complete independent double review): {len(reliability_item_ids)}")

    # Main analyses.
    print("\nCalculating reliability statistics...")
    reliability = build_reliability_by_dimension(judgments, dimensions, reliability_item_ids)
    label_distribution = build_label_distribution(judgments, dimensions, reliability_item_ids)
    disagreement_patterns = build_disagreement_patterns(judgments, dimensions, reliability_item_ids)

    item_level = build_item_level(items, judgments, dimensions, adjudicated_ids)
    strict_summary = build_strict_item_agreement_summary(item_level)
    global_summary = build_global_agreement_summary(reliability)

    annotator_stats = build_annotator_stats(judgments, dimensions, final_item_ids)
    reviewer_pair_stats = build_reviewer_pair_stats(judgments, dimensions, reliability_item_ids)
    loo_detail, loo_summary = build_leave_one_annotator_out(judgments, dimensions, final_item_ids)

    by_axis = build_group_reliability(items, judgments, dimensions, reliability_item_ids, "Axe")
    by_difficulty = build_group_reliability(items, judgments, dimensions, reliability_item_ids, "Difficulty")

    adjud_summary, required_ids, matched_ids, missing_required = build_adjudication_summary(items, adjudication)
    validation_funnel = build_validation_funnel(items, judgments, item_level, adjudication, dimensions)
    why_flagged = build_why_flagged_summary(adjudication, dimensions)

    # Required-adjudication cases that are not in the senior file are important.
    if missing_required:
        missing_rows = pd.DataFrame([
            {
                "scope": "adjudication",
                "item_id": item_id,
                "annotator_id": "",
                "issue": "Item marked as requiring senior adjudication but absent from adjudication file",
            }
            for item_id in sorted(missing_required)
        ])
        quality = pd.concat([quality, missing_rows], ignore_index=True).drop_duplicates()

    paper_main = build_paper_main_table(reliability)
    paper_dataset = build_paper_dataset_summary(
        items, judgments, reliability_item_ids, adjud_summary, strict_summary
    )
    readme = build_readme(dimensions)

    print("\nWriting publication-oriented workbook...")
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        readme.to_excel(writer, index=False, sheet_name="README")
        paper_dataset.to_excel(writer, index=False, sheet_name="PAPER_DATASET_SUMMARY")
        paper_main.to_excel(writer, index=False, sheet_name="PAPER_MAIN_TABLE")
        validation_funnel.to_excel(writer, index=False, sheet_name="VALIDATION_FUNNEL")
        strict_summary.to_excel(writer, index=False, sheet_name="STRICT_ITEM_AGREEMENT")
        global_summary.to_excel(writer, index=False, sheet_name="GLOBAL_AGREEMENT")
        reliability.to_excel(writer, index=False, sheet_name="RELIABILITY_BY_DIM")
        label_distribution.to_excel(writer, index=False, sheet_name="LABEL_PREVALENCE")
        disagreement_patterns.to_excel(writer, index=False, sheet_name="DISAGREEMENT_PATTERNS")
        adjud_summary.to_excel(writer, index=False, sheet_name="ADJUDICATION_SUMMARY")
        why_flagged.to_excel(writer, index=False, sheet_name="WHY_FLAGGED")
        annotator_stats.to_excel(writer, index=False, sheet_name="BY_ANNOTATOR")
        loo_summary.to_excel(writer, index=False, sheet_name="LOO_ROBUSTNESS_SUMMARY")
        loo_detail.to_excel(writer, index=False, sheet_name="LOO_SENSITIVITY")
        reviewer_pair_stats.to_excel(writer, index=False, sheet_name="REVIEWER_PAIR_STATS")
        by_axis.to_excel(writer, index=False, sheet_name="RELIABILITY_BY_AXIS")
        by_difficulty.to_excel(writer, index=False, sheet_name="RELIABILITY_BY_DIFF")
        item_level.to_excel(writer, index=False, sheet_name="ITEM_LEVEL_AUDIT")
        adjudication.to_excel(writer, index=False, sheet_name="SENIOR_ADJUDICATION")
        quality.to_excel(writer, index=False, sheet_name="DATA_QUALITY_AUDIT")

    format_output_workbook(OUTPUT_FILE)

    # Console summary with scientifically correct naming.
    strict_exact = strict_summary.loc[
        strict_summary["metric"] == "Strict exact unanimity on every dimension", "percent"
    ]
    strict_binary = strict_summary.loc[
        strict_summary["metric"] == "Strict clean-vs-issue unanimity on every dimension", "percent"
    ]
    adj_dict = dict(zip(adjud_summary["metric"], adjud_summary["value"]))

    print("\n" + "=" * 88)
    print("ANALYSIS COMPLETE")
    print("=" * 88)
    print(f"Final benchmark items                                  : {len(items)}")
    print(f"Primary independent judgments                          : {len(judgments)}")
    print(f"Observed validators                                     : {judgments['annotator_id'].nunique()}")
    print(f"IAA cohort: complete independent double review          : {len(reliability_item_ids)}")
    print(f"IAA cohort coverage of final benchmark                  : {safe_pct(len(reliability_item_ids), len(items))}%")
    print(f"Strict exact all-dimension unanimity in IAA cohort      : {strict_exact.iloc[0] if not strict_exact.empty else np.nan}%")
    print(f"Strict clean-vs-issue unanimity in IAA cohort           : {strict_binary.iloc[0] if not strict_binary.empty else np.nan}%")
    print(f"Items requiring senior adjudication                     : {int(adj_dict.get('items_requiring_senior_adjudication_n', 0))}/{len(items)} ({adj_dict.get('adjudication_rate_percent', np.nan)}%)")
    print(f"Senior adjudication coverage                            : {adj_dict.get('adjudication_coverage_percent', np.nan)}%")

    display = [
        "dimension",
        "reliability_items_n",
        "exact_observed_agreement_percent",
        "gwet_ac1_nominal",
        "krippendorff_alpha_nominal",
        "clean_vs_issue_observed_agreement_percent",
        "gwet_ac1_clean_vs_issue",
        "gwet_ac2_linear",
        "krippendorff_alpha_ordinal",
        "clean_rating_prevalence_percent",
    ]
    print("\nPaper-oriented reliability table:\n")
    print(reliability[display].to_string(index=False))

    if not quality.empty:
        print(f"\nData-quality audit rows to inspect                      : {len(quality)}")
    else:
        print("\nData-quality audit                                       : no issues detected")

    print(f"\nOutput written to:\n{OUTPUT_FILE.resolve()}")
    print("=" * 88)


main()


MOROCCAN OPEN-QUESTION BENCHMARK — RELIABILITY + ADJUDICATION ANALYSIS

Sheets in merged workbook:
  - README
  - OPEN_QUESTION_ITEMS
  - ADJUDICATION_QUEUE
  - ALL_JUDGMENTS
  - DATA_ISSUES

Detected dimensions:
  - linguistic_quality
  - answer_relevance
  - answer_completeness
  - factual_precision
  - safety_alignment

Cohorts
-------
Final benchmark items                         : 620
Primary independent judgments                 : 1268
Validators observed                            : 10
IAA cohort (complete independent double review): 603

Calculating reliability statistics...

Writing publication-oriented workbook...

ANALYSIS COMPLETE
Final benchmark items                                  : 620
Primary independent judgments                          : 1268
Observed validators                                     : 10
IAA cohort: complete independent double review          : 603
IAA cohort coverage of final benchmark                  : 97.26%
Strict exact all-dimension unanimity i